In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import subprocess

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
tmp_order = ['Start', 'Sendai', 'Skin', 'Inflamed', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2','IPS']

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
dmrs = pd.read_csv(f'{indir}/dmrs/methylpy/rna_cluster/rna_cluster_rms_results_collapsed.tsv',
                   header=0, sep='\t')
dmrs.shape

In [ ]:
dmrs.head()

In [ ]:
dmrs['hypomethylated_samples'].value_counts()

In [ ]:
dmrs.columns = [x.replace('methylation_level_', '') for x in dmrs.columns]
dmrs.columns 

In [ ]:
final_order = ['#chr', 'start', 'end', 'number_of_dms', 'hypermethylated_samples',
           'hypomethylated_samples'] + tmp_order
dmrs = dmrs.loc[:,final_order]
dmrs.shape

In [ ]:
tmp_dmrs = dmrs.iloc[:,:3]
tmp_dmrs['chrom_num'] = [int(x.split('chr')[1]) for x in tmp_dmrs['#chr']]
final_dmrs = tmp_dmrs.sort_values(by=['chrom_num', 'start'])
final_dmrs.shape

In [ ]:
final_dmrs.head()

In [ ]:
final_dmrs.tail()

In [ ]:
out = f'{projdir}/csv/dmrs/rna_cluster/dmrs.bed'
final_dmrs.to_csv(out, sep='\t', header=False, index=False)
subprocess.run(["bgzip", '-f', out], check=True)
subprocess.run(["tabix", '-f', "-p", "bed", f"{out}.gz"], check=True)

In [ ]:
out = f'{projdir}/csv/dmrs/rna_cluster/dmrs.header.bed'
dmrs.to_csv(out, sep='\t')

### explode

In [ ]:
dmrs['hypermethylated_samples'].fillna('', inplace=True)
dmrs['hypomethylated_samples'].fillna('', inplace=True)

In [ ]:
dmrs['hypermethylated_samples'] = dmrs['hypermethylated_samples'].str.split(',')
dmrs['hypomethylated_samples'] = dmrs['hypomethylated_samples'].str.split(',')
dmrs = dmrs.explode('hypermethylated_samples').reset_index(drop=True)
dmrs = dmrs.explode('hypomethylated_samples').reset_index(drop=True)
dmrs.index = [f'{x}_{y}' for x,y in zip(dmrs['#chr'], dmrs['start'])]
dmrs.shape

In [ ]:
s = 'Start'
dmrs[dmrs['hypomethylated_samples'] == s]

In [ ]:
dmrs.columns

In [ ]:
outdir = f'{projdir}/csv/dmrs/rna_cluster/by_group'

def write_dmrs(s):
    hypo_dmrs = dmrs[dmrs['hypomethylated_samples'] == s]
    hyper_dmrs = dmrs[dmrs['hypermethylated_samples'] == s]
    hypo_dmrs = hypo_dmrs[~hypo_dmrs.index.duplicated()]
    hyper_dmrs = hyper_dmrs[~hyper_dmrs.index.duplicated()]
    hypo_dmrs.to_csv(f'{outdir}/{s}_hypo.csv', sep='\t', header=None, index=None)
    hyper_dmrs.to_csv(f'{outdir}/{s}_hyper.csv', sep='\t', header=None, index=None)
    
    final_hypo = hypo_dmrs[hypo_dmrs['number_of_dms'] > 1]
    final_hyper = hyper_dmrs[hyper_dmrs['number_of_dms'] > 1]

    final_hypo.to_csv(f'{outdir}/{s}_hypo_dms1.csv', sep='\t', header=None, index=None)
    final_hyper.to_csv(f'{outdir}/{s}_hyper_dms1.csv', sep='\t', header=None, index=None)

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(write_dmrs, tmp_order), total=len(tmp_order)))

In [ ]:
!date